## Verstappen vs Field — Focused Comparisons

This companion notebook reuses the exported data from the main Belgian GP analysis and builds **clean, pairwise comparisons** between Max Verstappen (VER) and each of the other three drivers (PER, SAI, RUS).

It assumes you have already run the main notebook so that these files exist in the `./data` folder:
- `laps_data.csv`
- `session_results.csv`
- `telemetry_VER.csv`, `telemetry_PER.csv`, `telemetry_SAI.csv`, `telemetry_RUS.csv`

This notebook creates extra charts only:
- Tire degradation (linear regression): VER vs PER, VER vs SAI, VER vs RUS
- Pearson correlation (tire age vs lap time): VER vs each driver
- Telemetry speed traces: VER vs each driver
- Telemetry throttle/brake traces: VER vs each driver

Your original notebook and its all‑four‑driver charts remain unchanged.

In [ ]:
import os                         # Work with folders and file paths
import numpy as np               # Numerical operations and arrays
import pandas as pd              # DataFrames
import matplotlib.pyplot as plt  # Plotting
from scipy import stats          # Regression and correlation tools

plt.style.use("dark_background")  # Match style from main notebook

DATA_DIR = "./data"
CHARTS_DIR = "./charts"
os.makedirs(CHARTS_DIR, exist_ok=True)

# Fixed driver list and colors (same as main notebook)
DRIVERS = ["VER", "PER", "SAI", "RUS"]
DRIVER_COLORS = {
    "VER": "#E10600",
    "PER": "#0066CC",
    "SAI": "#FF8000",
    "RUS": "#C0C0C0",
}

# -------- Helper: clean laps in the same way as Sections 3/5 --------
def load_clean_laps() -> pd.DataFrame:
    """Load laps_data.csv and apply the same 'clean racing laps' filters
    used in the main notebook (pit in/out removed, lap 1 removed, >2 SD slow laps removed).
    """
    laps_path = os.path.join(DATA_DIR, "laps_data.csv")
    laps = pd.read_csv(laps_path)

    # Ensure numeric types
    laps["LapNumber"] = pd.to_numeric(laps["LapNumber"], errors="coerce")

    # Ensure LapTimeSeconds exists; if not, convert from LapTime
    if "LapTimeSeconds" not in laps.columns and "LapTime" in laps.columns:
        laps["LapTime"] = pd.to_timedelta(laps["LapTime"], errors="coerce")
        laps["LapTimeSeconds"] = laps["LapTime"].dt.total_seconds()

    laps = laps.dropna(subset=["LapTimeSeconds"])

    # Standardize pit time columns, then remove pit in/out laps
    for col in ["PitInTime", "PitOutTime"]:
        if col in laps.columns:
            laps[col] = laps[col].replace(["", "NaT", "nan"], np.nan)

    if "PitInTime" in laps.columns:
        laps = laps[laps["PitInTime"].isna()]
    if "PitOutTime" in laps.columns:
        laps = laps[laps["PitOutTime"].isna()]

    # Remove first lap (Lap 1)
    laps = laps[laps["LapNumber"] != 1]

    # Only our four drivers
    laps = laps[laps["Driver"].isin(DRIVERS)]

    # Remove laps > 2 SD slower than each driver's mean
    per_driver = laps.groupby("Driver")["LapTimeSeconds"].agg(["mean", "std"]).rename(
        columns={"mean": "driver_mean", "std": "driver_std"}
    )
    laps = laps.merge(per_driver, left_on="Driver", right_index=True)
    laps["z_score"] = (laps["LapTimeSeconds"] - laps["driver_mean"]) / laps["driver_std"]
    clean = laps[laps["z_score"] <= 2].copy()
    clean = clean.drop(columns=["driver_mean", "driver_std", "z_score"])

    return clean


# -------- Helper: build TyreLife vs LapTimeSeconds dataset --------
def build_regression_data(clean_laps: pd.DataFrame) -> pd.DataFrame:
    """Prepare TyreLife vs LapTimeSeconds for the four drivers, removing rows with missing values."""
    df = clean_laps.copy()
    df["TyreLife"] = pd.to_numeric(df.get("TyreLife"), errors="coerce")
    df["LapTimeSeconds"] = pd.to_numeric(df["LapTimeSeconds"], errors="coerce")
    df = df.dropna(subset=["TyreLife", "LapTimeSeconds"])
    df = df[df["Driver"].isin(DRIVERS)]
    return df


# -------- Helper: run simple linear regression (slope, intercept) --------
def run_linreg(x: np.ndarray, y: np.ndarray):
    """Run a simple linear regression (y ~ x) using scipy.stats.linregress."""
    lr = stats.linregress(x, y)
    return lr.slope, lr.intercept


# -------- Helper: Spa annotations (same as main notebook) --------
SPA_MARKERS = [
    (1000, "Eau Rouge / Raidillon"),
    (2000, "Kemmel Straight"),
    (4200, "Les Combes"),
    (6500, "Bus Stop Chicane"),
]


def add_spa_annotations(ax):
    """Add vertical dashed lines + labels for key Spa sections."""
    for x_m, label in SPA_MARKERS:
        ax.axvline(x_m, color="white", linestyle="--", alpha=0.4, linewidth=1.0)
        ax.text(
            x_m,
            0.98,  # y in axes coords (top)
            label,
            transform=ax.get_xaxis_transform(),
            rotation=90,
            va="top",
            ha="right",
            fontsize=9,
            color="white",
            alpha=0.85,
        )

In [ ]:
# -------- Pairwise tire degradation plots: VER vs PER/SAI/RUS --------

clean_laps = load_clean_laps()
regression_data = build_regression_data(clean_laps)

pairs = [("PER", "ver_vs_per"), ("SAI", "ver_vs_sai"), ("RUS", "ver_vs_rus")]

for other, tag in pairs:
    subset = regression_data[regression_data["Driver"].isin(["VER", other])]

    fig, ax = plt.subplots(figsize=(8, 5))

    for driver in ["VER", other]:
        df_d = subset[subset["Driver"] == driver]
        x = df_d["TyreLife"].to_numpy(float)
        y = df_d["LapTimeSeconds"].to_numpy(float)

        # Scatter points (raw laps)
        ax.scatter(
            x,
            y,
            color=DRIVER_COLORS[driver],
            alpha=0.4,
            s=30,
            label=driver,
        )

        # Only fit a line if we have enough points
        if len(x) >= 3:
            slope, intercept = run_linreg(x, y)
            x_line = np.array([x.min(), x.max()])
            y_line = slope * x_line + intercept

            ax.plot(
                x_line,
                y_line,
                color=DRIVER_COLORS[driver],
                linewidth=2,
            )

            ax.text(
                x_line.max(),
                y_line[-1],
                f"{driver}: {slope:+.2f}s/lap",
                color=DRIVER_COLORS[driver],
                fontsize=9,
                ha="left",
                va="center",
            )

    ax.set_title(f"Tire Degradation — VER vs {other} (Linear Regression)")
    ax.set_xlabel("Tire Age (laps)")
    ax.set_ylabel("Lap Time (seconds)")
    ax.grid(alpha=0.2, linestyle="--")

    ax.text(
        1.0,
        0.0,
        "Data: FastF1 | 2022 Belgian GP",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=8,
        color="white",
        alpha=0.7,
    )

    out_path = os.path.join(CHARTS_DIR, f"tire_regression_{tag}.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved {out_path}")

    plt.close(fig)

In [ ]:
# -------- Pairwise Pearson correlation: VER vs each driver --------

corr_rows = []

for driver in DRIVERS:
    df_d = regression_data[regression_data["Driver"] == driver]
    x = df_d["TyreLife"].to_numpy(float)
    y = df_d["LapTimeSeconds"].to_numpy(float)

    if len(x) < 3:
        corr_rows.append({"Driver": driver, "r": np.nan, "p": np.nan})
        continue

    r, p = stats.pearsonr(x, y)
    corr_rows.append({"Driver": driver, "r": r, "p": p})

corr_df = pd.DataFrame(corr_rows).set_index("Driver")

pairs = [("PER", "ver_vs_per"), ("SAI", "ver_vs_sai"), ("RUS", "ver_vs_rus")]

for other, tag in pairs:
    sub = corr_df.loc[["VER", other]].copy()

    fig, ax = plt.subplots(figsize=(7, 3.5))

    drivers_pair = ["VER", other]
    colors_pair = [DRIVER_COLORS[d] for d in drivers_pair]
    r_vals = sub["r"].values

    bars = ax.barh(drivers_pair, r_vals, color=colors_pair, alpha=0.9)

    ax.axvline(0, color="white", linestyle="--", linewidth=1.2, alpha=0.9)

    for bar, r_val in zip(bars, r_vals):
        x_text = bar.get_width()
        y_text = bar.get_y() + bar.get_height() / 2
        offset = 0.02 if x_text >= 0 else -0.02
        ha = "left" if x_text >= 0 else "right"

        ax.text(
            x_text + offset,
            y_text,
            f"r = {r_val:.2f}",
            va="center",
            ha=ha,
            color="white",
            fontsize=9,
        )

    ax.set_title(f"Pearson Correlation — VER vs {other} (TyreLife vs LapTime)")
    ax.set_xlabel("Pearson r (correlation coefficient)")
    ax.grid(alpha=0.2, linestyle="--", axis="x")

    ax.text(
        0.99,
        0.05,
        "→ Higher r = tires affected pace more",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=9,
        color="white",
        alpha=0.85,
    )

    out_path = os.path.join(CHARTS_DIR, f"pearson_{tag}.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved {out_path}")

    plt.close(fig)

In [ ]:
# -------- Pairwise telemetry speed traces: VER vs each driver --------


def load_telemetry() -> dict:
    """Load telemetry CSVs for the four drivers into a dict."""
    tel = {}
    for d in DRIVERS:
        path = os.path.join(DATA_DIR, f"telemetry_{d}.csv")
        if not os.path.exists(path):
            print(f"WARNING: Missing telemetry file {path}; skipping {d}.")
            continue
        df = pd.read_csv(path)
        if "Distance" not in df.columns or "Speed" not in df.columns:
            print(f"WARNING: {path} missing 'Distance' or 'Speed'; skipping {d}.")
            continue
        df["Distance"] = pd.to_numeric(df["Distance"], errors="coerce")
        df["Speed"] = pd.to_numeric(df["Speed"], errors="coerce")
        df = df.dropna(subset=["Distance", "Speed"]).sort_values("Distance")
        tel[d] = df
    return tel


telemetry = load_telemetry()

# Build a common distance grid per pair (to keep things independent)
for other, tag in [("PER", "ver_vs_per"), ("SAI", "ver_vs_sai"), ("RUS", "ver_vs_rus")]:
    if "VER" not in telemetry or other not in telemetry:
        print(f"Skipping VER vs {other} (missing telemetry).")
        continue

    df_ver = telemetry["VER"]
    df_other = telemetry[other]

    min_common = max(df_ver["Distance"].min(), df_other["Distance"].min())
    max_common = min(df_ver["Distance"].max(), df_other["Distance"].max())
    if max_common <= min_common:
        print(f"Skipping VER vs {other}: no overlapping distance range.")
        continue

    distance_grid = np.linspace(min_common, max_common, 800)

    def interp_speed(df):
        dist = df["Distance"].to_numpy(float)
        spd = df["Speed"].to_numpy(float)
        return np.interp(distance_grid, dist, spd)

    speed_ver = interp_speed(df_ver)
    speed_other = interp_speed(df_other)

    fig, ax = plt.subplots(figsize=(12, 4.5))

    ax.plot(distance_grid, speed_ver, color=DRIVER_COLORS["VER"], linewidth=1.5, alpha=0.9, label="VER")
    ax.plot(distance_grid, speed_other, color=DRIVER_COLORS[other], linewidth=1.5, alpha=0.9, label=other)

    add_spa_annotations(ax)

    ax.set_title(f"Speed Trace — VER vs {other} (Fastest Lap)")
    ax.set_xlabel("Distance (m)")
    ax.set_ylabel("Speed (km/h)")
    ax.grid(alpha=0.2, linestyle="--")
    ax.legend(loc="upper right", title="Driver")

    ax.text(
        1.0,
        0.0,
        "Data: FastF1 | 2022 Belgian GP",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=8,
        color="white",
        alpha=0.7,
    )

    out_path = os.path.join(CHARTS_DIR, f"telemetry_speed_{tag}.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved {out_path}")

    plt.close(fig)

In [ ]:
# -------- Pairwise telemetry throttle & brake: VER vs each driver --------

for other, tag in [("PER", "ver_vs_per"), ("SAI", "ver_vs_sai"), ("RUS", "ver_vs_rus")]:
    if "VER" not in telemetry or other not in telemetry:
        print(f"Skipping VER vs {other} (missing telemetry).")
        continue

    df_ver = telemetry["VER"]
    df_other = telemetry[other]

    # Require Throttle and Brake columns
    for df, name in [(df_ver, "VER"), (df_other, other)]:
        if "Throttle" not in df.columns or "Brake" not in df.columns:
            print(f"Skipping VER vs {other}: missing Throttle/Brake for {name}.")
            break
    else:
        # Build common distance grid for this pair
        min_common = max(df_ver["Distance"].min(), df_other["Distance"].min())
        max_common = min(df_ver["Distance"].max(), df_other["Distance"].max())
        if max_common <= min_common:
            print(f"Skipping VER vs {other}: no overlapping distance range.")
            continue

        distance_grid = np.linspace(min_common, max_common, 800)

        def interp_series(df, col):
            dist = df["Distance"].to_numpy(float)
            vals = pd.to_numeric(df[col], errors="coerce").to_numpy(float)
            mask = ~np.isnan(vals)
            return np.interp(distance_grid, dist[mask], vals[mask])

        thr_ver = interp_series(df_ver, "Throttle")
        thr_other = interp_series(df_other, "Throttle")

        # Brake may be boolean or 0/1; convert to numeric
        def to_numeric_brake(series):
            return series.replace(
                {True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0}
            )

        df_ver_br = df_ver.copy()
        df_other_br = df_other.copy()
        df_ver_br["Brake"] = to_numeric_brake(df_ver_br["Brake"])
        df_other_br["Brake"] = to_numeric_brake(df_other_br["Brake"])

        br_ver = interp_series(df_ver_br, "Brake")
        br_other = interp_series(df_other_br, "Brake")

        fig, (ax_thr, ax_brk) = plt.subplots(
            nrows=2,
            ncols=1,
            figsize=(12, 6),
            sharex=True,
            gridspec_kw={"height_ratios": [2, 1]},
        )

        # Throttle subplot
        ax_thr.plot(distance_grid, thr_ver, color=DRIVER_COLORS["VER"], linewidth=1.5, alpha=0.9, label="VER")
        ax_thr.plot(distance_grid, thr_other, color=DRIVER_COLORS[other], linewidth=1.5, alpha=0.9, label=other)
        add_spa_annotations(ax_thr)
        ax_thr.set_title(f"Throttle Application — VER vs {other}")
        ax_thr.set_ylabel("Throttle (%)")
        ax_thr.set_ylim(-5, 105)
        ax_thr.grid(alpha=0.2, linestyle="--")
        ax_thr.legend(loc="upper right", title="Driver")

        # Brake subplot
        ax_brk.plot(distance_grid, br_ver, color=DRIVER_COLORS["VER"], linewidth=1.5, alpha=0.9, label="VER")
        ax_brk.plot(distance_grid, br_other, color=DRIVER_COLORS[other], linewidth=1.5, alpha=0.9, label=other)
        add_spa_annotations(ax_brk)
        ax_brk.set_title("Brake Application")
        ax_brk.set_xlabel("Distance (m)")
        ax_brk.set_ylabel("Brake (0/1)")
        ax_brk.set_ylim(-0.1, 1.1)
        ax_brk.grid(alpha=0.2, linestyle="--")

        fig.suptitle(f"Telemetry — VER vs {other} (Fastest Lap)", fontsize=14, y=0.98)

        out_path = os.path.join(CHARTS_DIR, f"telemetry_throttle_brake_{tag}.png")
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        print(f"Saved {out_path}")

        plt.close(fig)